In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
from pathlib import Path
sys.path.append('../')
from utils import *  # key functions for this project

THRESHOLD = 50


### Get ground-truth looping probabilities

In [ ]:
loop_nums = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8])
looping_probs = pd.Series(index=loop_nums, dtype='float', name='looping_prob')
looping_probs.index.name = 'loop_num'
n_reps = 19 - 10 + 1  # 10..19 inclusive

## -- CTCF loops: as defined by cohesin positions --

loop_nums_CTCF = np.array([0, 1, 2])
total_iters = len(loop_nums_CTCF) * n_reps

with tqdm(total=total_iters, desc="looping probabilities (CTCF)", unit="rep") as pbar:
    for loop_num in loop_nums_CTCF:
        looping_probs.loc[loop_num] = prob_loop_CTCF_all_reps(loop_num, pbar=pbar)
        

In [4]:
## -- EP loops: prob. under 50nm minus background --

loop_nums_EP = np.array([3, 4, 5])
total_iters = len(loop_nums_EP) * n_reps * 2  # extra factor of 2 for sticky vs. non-sticky

with tqdm(total=total_iters, desc="looping probabilities (EP)", unit="rep") as pbar:
    for loop_num in loop_nums_EP:
        
        prob_under_threshold_regular = prob_under_threshold_all_reps(loop_num, noise=0, non_sticky=False, threshold=THRESHOLD, pbar=pbar)
        prob_under_threshold_non_sticky = prob_under_threshold_all_reps(loop_num, noise=0, non_sticky=True, threshold=THRESHOLD, pbar=pbar)
        
        looping_prob_this_loop = prob_under_threshold_regular - prob_under_threshold_non_sticky
        
        looping_probs.loc[loop_num] = looping_prob_this_loop
        

looping probabilities (EP):  10%|█         | 6/60 [00:09<01:27,  1.61s/rep]


KeyboardInterrupt: 

In [ ]:
## -- random loops: prob. under 50nm --

loop_nums_random = np.array([6, 7, 8])
total_iters = len(loop_nums_random) * n_reps

with tqdm(total=total_iters, desc="looping probabilities (random)", unit="rep") as pbar:
    for loop_num in loop_nums_random:
        
        prob_under_threshold = prob_under_threshold_all_reps(loop_num, noise=0, threshold=THRESHOLD, pbar=pbar)
        
        looping_prob_this_loop = prob_under_threshold
        
        looping_probs.loc[loop_num] = looping_prob_this_loop
        

looping probabilities (random):   0%|          | 0/30 [00:00<?, ?rep/s]

In [ ]:
looping_probs.to_csv('../data/looping_probs.csv', float_format="%.6f")

### Get probabilities below threshold

In [ ]:
noise_levels = np.array([0, 10, 20, 30, 40, 50])

probs_under_threshold = pd.DataFrame(index=loop_nums, columns=noise_levels, dtype='float')
probs_under_threshold.index.name = 'loop_num'
probs_under_threshold.columns.name = 'noise'
total_iters = len(loop_nums) * len(noise_levels) * n_reps

with tqdm(total=total_iters, desc="probability under threshold", unit="rep") as pbar:
    for loop_num in loop_nums:
        for noise in noise_levels:
            probs_under_threshold.loc[loop_num, noise] = prob_under_threshold_all_reps(loop_num, noise, non_sticky=False, threshold=THRESHOLD, pbar=pbar)
        

probability under threshold:   0%|          | 0/540 [00:00<?, ?rep/s]

In [ ]:
with open('../data/probs_under_threshold.csv', mode='w') as f:
    f.write('# Note: columns correspond to different noise levels (provided as sigma_x in nm)\n')
    probs_under_threshold.to_csv(f, float_format="%.6f") # Pass the file handler 'f' to to_csv


### Get 3D distance distributions

In [ ]:
rows = []  # counts of occurrences of each nonnegative integer: 0, 1, 2 nm...

total_iters = len(loop_nums) * len(noise_levels) * n_reps

with tqdm(total=total_iters, desc="gathering 3D distances", unit="rep") as pbar:
    for loop_num in loop_nums:
        for noise in noise_levels:
        
            dist_3D_distribution = dist_3D_distribution_all_reps(loop_num=loop_num, noise=noise, non_sticky=False, pbar=pbar)
            hist = np.bincount(np.round(dist_3D_distribution).astype('int'))
            
            rows.append(
                {
                    "loop": loop_num,
                    "noise": noise,
                    "hist": hist
                }
            )
            
dist_3D_distribution_histograms = pd.DataFrame(rows)


gathering 3D distances:   0%|          | 0/540 [00:00<?, ?rep/s]

In [ ]:
dist_3D_distribution_histograms.to_pickle('../data/3D_dist_histograms.pkl')

### Get 3D distance distributions for looped and unlooped states separately

In [ ]:
loop_nums_CTCF = np.array([0, 1, 2])
noise_levels = np.array([0, 10, 20, 30, 40, 50])
reps = np.arange(10,19+1)

rows = []  # counts of occurrences of each nonnegative integer: 0, 1, 2 nm...

total_iters = len(loop_nums_CTCF) * len(noise_levels) * len(reps)

with tqdm(total=total_iters, desc="gathering 3D distances", unit="rep") as pbar:
    for loop_num in loop_nums_CTCF:
        for noise in noise_levels:
            
            dist_3D_looped = []
            dist_3D_unlooped = []
                
            for rep in reps:              
                traj = load_trajectory(loop_num, rep, noise)
                ctcf_state = load_ctcf_state(loop_num, rep)
                dist_3D_looped += list(traj[ctcf_state==1])
                dist_3D_unlooped += list(traj[ctcf_state==0])
                
                pbar.update(1)
                
            hist_looped = np.bincount(np.round(np.array(dist_3D_looped)).astype('int'))
            hist_unlooped = np.bincount(np.round(np.array(dist_3D_unlooped)).astype('int'))
                
            rows.append(
                {
                    "loop": loop_num,
                    "noise": noise,
                    "hist_looped": hist_looped,
                    "hist_unlooped": hist_unlooped
                }
            )
            
dist_3D_distribution_histograms_with_looping = pd.DataFrame(rows)


gathering 3D distances:   0%|          | 0/180 [00:00<?, ?rep/s]

In [ ]:
dist_3D_distribution_histograms_with_looping.to_pickle('../data/3D_dist_histograms_with_looping.pkl')
